In [5]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import umap
import hdbscan
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import DataLoader, TensorDataset
from sentence_transformers import SentenceTransformer
import logging
from tqdm import tqdm
from collections import Counter
from warnings import filterwarnings

# Suppress FutureWarnings
filterwarnings('ignore', category=FutureWarning)

# ─── CONFIG ────────────────────────────────────────────────────────────────────
CONFIG = {
    'data_path': r"C:\Users\PC\Downloads\CinemateMovieDataset (1).csv",
    'embeddings_path': "embeddings.npy",
    'output_dir': "final_movie_clusters232323",
    
    # UMAP Settings
    'umap_components': 15,
    'umap_n_neighbors': 30,
    'umap_min_dist': 0.1,
    'umap_metric': 'cosine',
    
    # Autoencoder Settings
    'latent_dim': 64,
    'autoencoder_epochs': 30,
    'autoencoder_batch_size': 512,
    'autoencoder_lr': 0.001,
    
    # HDBSCAN Settings
    'hdb_min_cluster_size': [40, 60],
    'hdb_min_samples': [1, 3],
    'hdb_eps': [0.1, 0.2],
    'hdb_method': 'eom',
    'hdb_cores': -1,
    
    # Post-processing
    'min_cluster_size': 75,
    'noise_reassign_threshold': 0.80,
    'pca_variance': 0.95,
    
    # General
    'variance_threshold': 0.01,
    'silhouette_sample_size': 5000,
    'use_hybrid_ae': True
}

# Setup
os.makedirs(CONFIG['output_dir'], exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ─── DATA LOADING ─────────────────────────────────────────────────────────────
def load_data():
    """Optimized data loading with modern datetime parsing"""
    df = pd.read_csv(CONFIG['data_path'])
    df = df.rename(columns={
        'TMDBId': 'MovieID',
        'Title': 'title',
        'ReleaseDate': 'release_date',
        'Runtime': 'runtime'
    })
    
    # Modern datetime parsing
    df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
    df['release_year'] = df['release_year'].fillna(df['release_year'].median())
    
    # Numeric conversion
    df['runtime'] = pd.to_numeric(df['runtime'], errors='coerce')
    df['runtime'] = df['runtime'].fillna(df['runtime'].median())
    
    # Filter and deduplicate
    df = df[df['overview'].notna() & (df['overview'] != '')]
    movie_info = df.groupby('MovieID').first()[['title', 'release_year', 'runtime', 'overview']]
    
    # Genre processing
    genre_df = df[['MovieID', 'GenreId']].drop_duplicates()
    genre_pivot = pd.crosstab(genre_df.MovieID, genre_df.GenreId)
    genre_pivot = genre_pivot.reindex(movie_info.index, fill_value=0)
    genre_pivot.columns = [f"Genre:{g}" for g in genre_pivot.columns]
    
    return movie_info.join(genre_pivot)

# ─── EMBEDDINGS ───────────────────────────────────────────────────────────────
def get_embeddings(df):
    """Cache-aware embedding generation"""
    if os.path.exists(CONFIG['embeddings_path']):
        emb = np.load(CONFIG['embeddings_path'])
        if emb.shape[0] == len(df):
            return emb
    
    model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
    emb = model.encode(
        df['overview'].tolist(),
        batch_size=CONFIG['autoencoder_batch_size'],
        show_progress_bar=True
    )
    np.save(CONFIG['embeddings_path'], emb)
    return emb

# ─── DIMENSIONALITY REDUCTION ────────────────────────────────────────────────
def reduce_dims(emb):
    """Optimized UMAP projection"""
    reducer = umap.UMAP(
        n_components=CONFIG['umap_components'],
        n_neighbors=CONFIG['umap_n_neighbors'],
        min_dist=CONFIG['umap_min_dist'],
        metric=CONFIG['umap_metric'],
        n_jobs=-1,
        verbose=False
    )
    return reducer.fit_transform(emb)

# ─── AUTOENCODER ─────────────────────────────────────────────────────────────
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )
    
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

def train_ae(features):
    """Optimized autoencoder training"""
    input_dim = features.shape[1]
    model = Autoencoder(input_dim, CONFIG['latent_dim']).to(device)
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['autoencoder_lr'])
    
    tensor_data = torch.tensor(features, dtype=torch.float32)
    dataloader = DataLoader(
        TensorDataset(tensor_data),
        batch_size=CONFIG['autoencoder_batch_size'],
        shuffle=True,
        pin_memory=True
    )
    
    best_loss = float('inf')
    for epoch in tqdm(range(CONFIG['autoencoder_epochs']), desc="Training AE"):
        model.train()
        epoch_loss = 0
        for (batch,) in dataloader:
            batch = batch.to(device)
            optimizer.zero_grad()
            recon, _ = model(batch)
            loss = nn.MSELoss()(recon, batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save(model.state_dict(), os.path.join(CONFIG['output_dir'], 'ae_weights.pt'))
    
    model.load_state_dict(torch.load(os.path.join(CONFIG['output_dir'], 'ae_weights.pt')))
    with torch.no_grad():
        return model.encoder(tensor_data.to(device)).cpu().numpy()

# ─── CLUSTERING ───────────────────────────────────────────────────────────────
def hdbscan_cluster(features):
    """Core HDBSCAN clustering with parameter search"""
    best_score = -1
    best_labels = None
    
    clusterer = hdbscan.HDBSCAN(
        core_dist_n_jobs=CONFIG['hdb_cores'],
        cluster_selection_method=CONFIG['hdb_method'],
        memory=os.path.join(CONFIG['output_dir'], 'hdbscan_cache')
    )
    
    for min_size in CONFIG['hdb_min_cluster_size']:
        for min_samples in CONFIG['hdb_min_samples']:
            for eps in CONFIG['hdb_eps']:
                clusterer.set_params(
                    min_cluster_size=min_size,
                    min_samples=min_samples,
                    cluster_selection_epsilon=eps
                )
                labels = clusterer.fit_predict(features)
                valid_mask = labels != -1
                
                if valid_mask.sum() > 10:
                    if len(features) > CONFIG['silhouette_sample_size']:
                        sample_idx = np.random.choice(
                            np.where(valid_mask)[0],
                            size=CONFIG['silhouette_sample_size'],
                            replace=False
                        )
                        score = silhouette_score(features[sample_idx], labels[sample_idx])
                    else:
                        score = silhouette_score(features[valid_mask], labels[valid_mask])
                    
                    if score > best_score:
                        best_score = score
                        best_labels = labels
    
    logging.info(f"Best silhouette score: {best_score:.4f}")
    return best_labels

def reassign_noise(labels, features):
    """Fixed noise reassignment function"""
    valid_mask = labels != -1
    if valid_mask.sum() == 0:
        return labels
    
    valid_indices = np.where(valid_mask)[0]
    nbrs = NearestNeighbors(n_neighbors=5).fit(features[valid_mask])
    
    noise_indices = np.where(labels == -1)[0]
    distances, neighbor_indices = nbrs.kneighbors(features[noise_indices])
    
    for i, (dists, nb_indices) in enumerate(zip(distances, neighbor_indices)):
        neighbor_labels = labels[valid_indices[nb_indices]]
        if len(set(neighbor_labels)) == 1 and (dists < CONFIG['noise_reassign_threshold']).any():
            labels[noise_indices[i]] = neighbor_labels[0]
    
    return labels

def postprocess_clusters(labels):
    """Merge small clusters into noise"""
    counts = Counter(labels)
    small_clusters = [k for k,v in counts.items() if v < CONFIG['min_cluster_size'] and k != -1]
    return np.where(np.isin(labels, small_clusters), -1, labels)

# ─── ANALYSIS & VISUALIZATION ────────────────────────────────────────────────
def analyze_clusters(df):
    """Generate cluster characteristics"""
    results = []
    for cluster in df['cluster'].unique():
        if cluster == -1: continue
        subset = df[df['cluster'] == cluster]
        
        # Top genres
        genre_cols = [c for c in df.columns if c.startswith('Genre:')]
        top_genres = subset[genre_cols].sum().nlargest(5)
        
        # Year stats
        year_median = subset['release_year'].median()
        
        results.append({
            'Cluster': cluster,
            'Size': len(subset),
            'Top Genres': ', '.join([g.split(':')[1] for g in top_genres.index]),
            'Median Year': int(year_median)
        })
    
    return pd.DataFrame(results).sort_values('Size', ascending=False)

def visualize_results(features, labels):
    """Generate diagnostic plots"""
    plt.figure(figsize=(12,5))
    
    # Cluster size distribution
    counts = Counter(labels)
    cluster_sizes = [v for k,v in counts.items() if k != -1]
    
    plt.subplot(121)
    plt.hist(cluster_sizes, bins=50)
    plt.title(f'Cluster Sizes (>{CONFIG["min_cluster_size"]} movies)')
    plt.xlabel('Size')
    plt.ylabel('Count')
    
    # t-SNE visualization
    from sklearn.manifold import TSNE
    sample_size = min(5000, len(features))
    idx = np.random.choice(len(features), sample_size, replace=False)
    embed = TSNE(n_components=2).fit_transform(features[idx])
    
    plt.subplot(122)
    plt.scatter(embed[:,0], embed[:,1], c=labels[idx], cmap='Spectral', s=1)
    plt.colorbar()
    plt.title('t-SNE Cluster Visualization')
    
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['output_dir'], 'cluster_diagnostics.png'))
    plt.close()

# ─── MAIN PIPELINE ────────────────────────────────────────────────────────────
def main():
    # 1. Load data
    logging.info("Loading data...")
    df = load_data()
    
    # 2. Generate embeddings
    logging.info("Generating embeddings...")
    embeddings = get_embeddings(df)
    
    # 3. Dimensionality reduction
    logging.info("Reducing dimensions...")
    reduced_emb = reduce_dims(embeddings)
    reduced_emb = MinMaxScaler().fit_transform(reduced_emb)
    
    # 4. Create hybrid features
    logging.info("Creating hybrid features...")
    genre_cols = [c for c in df.columns if c.startswith('Genre:')]
    genre_features = df[genre_cols].values
    idf = np.log1p(len(df) / (genre_features.sum(axis=0) + 1))
    genre_features = genre_features * idf
    
    numeric_features = MinMaxScaler().fit_transform(df[['release_year', 'runtime']])
    hybrid_features = np.hstack([genre_features, numeric_features, reduced_emb])
    hybrid_features = VarianceThreshold(CONFIG['variance_threshold']).fit_transform(hybrid_features)
    
    if CONFIG['use_hybrid_ae']:
        hybrid_features = train_ae(hybrid_features)
    
    # 5. Cluster data with PCA
    logging.info("Clustering...")
    hybrid_features = PCA(n_components=CONFIG['pca_variance']).fit_transform(hybrid_features)
    labels = hdbscan_cluster(hybrid_features)
    labels = reassign_noise(labels, hybrid_features)
    labels = postprocess_clusters(labels)
    
    # 6. Save and analyze results
    logging.info("Saving results...")
    df['cluster'] = labels
    output_path = os.path.join(CONFIG['output_dir'], 'clustered_movies.csv')
    df.to_csv(output_path, index=False)
    
    # Cluster analysis
    cluster_stats = analyze_clusters(df)
    cluster_stats.to_csv(os.path.join(CONFIG['output_dir'], 'cluster_analysis.csv'), index=False)
    
    # Visualization
    visualize_results(hybrid_features, labels)
    
    # Final stats
    cluster_counts = df['cluster'].value_counts()
    noise_percent = (df['cluster'] == -1).mean() * 100
    meaningful_clusters = len(cluster_counts) - 1
    
    logging.info(f"\n=== FINAL RESULTS ===")
    logging.info(f"Meaningful clusters: {meaningful_clusters}")
    logging.info(f"Noise points: {noise_percent:.1f}%")
    logging.info("\nTop clusters by characteristics:\n" + str(cluster_stats.head()))

if __name__ == "__main__":
    main()

2025-06-08 17:52:20,803 - INFO - Loading data...
2025-06-08 17:52:23,069 - INFO - Generating embeddings...
2025-06-08 17:52:23,246 - INFO - Reducing dimensions...
2025-06-08 17:53:06,092 - INFO - Creating hybrid features...
Training AE: 100%|█████████████████████████████████████████████████████████████████████| 30/30 [00:19<00:00,  1.53it/s]
2025-06-08 17:53:25,865 - INFO - Clustering...
2025-06-08 17:53:58,669 - INFO - Best silhouette score: 0.7119
2025-06-08 17:53:58,935 - INFO - Saving results...
2025-06-08 17:54:11,958 - INFO - 
=== FINAL RESULTS ===
2025-06-08 17:54:11,959 - INFO - Meaningful clusters: 192
2025-06-08 17:54:11,959 - INFO - Noise points: 9.8%
2025-06-08 17:54:11,962 - INFO - 
Top clusters by characteristics:
    Cluster  Size             Top Genres  Median Year
15      180  9185     18, 12, 14, 16, 27         2007
11      203  6880     99, 12, 14, 16, 18         2013
80      205  6276     35, 12, 14, 16, 18         2004
36      204  3335     18, 35, 12, 14, 16      